In [311]:
# import libraries

In [312]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# import data

In [313]:
df_transactions = pd.read_csv(r"C:\Users\Care\Desktop\Dmart_Project\data\raw\Dmart_Consumer_Transactions_Raw.csv")
df_customers = pd.read_csv(r"C:\Users\Care\Desktop\Dmart_Project\data\raw\Dmart_Customer_Master_Raw.csv")
df_products = pd.read_csv(r"C:\Users\Care\Desktop\Dmart_Project\data\raw\Dmart_Product_Lookup.csv")

# Clan Transactions Data

## look the data

In [314]:
df_transactions.head()

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status
0,TX0000001,06-08-2025,DM002631,ST015,In-store,PR004,Sugar 1kg,Staples,1,48.0,0.00,Debit Card,Completed
1,TX0000002,30-08-2025,DM002236,ST008,App,PR005,Milk 1L,Dairy,2,64.0,0.02,Cash,Completed
2,TX0000003,14-11-2025,DM001854,ST023,In-store,PR002,Wheat Flour 5kg,STAPLES,4,245.0,0.02,Wallet,Completed
3,TX0000004,24-01-2025,DM000334,ST020,Online,PR020,LED Bulb,Home & Utility,1,85.0,0.02,Cash,Cancelled
4,TX0000005,19-06-2026,DM001319,ST002,In-store,PR012,Laundry Detergent,Home Care,1,299.0,0.15,UPI,Completed


In [315]:
df_transactions.shape

(15035, 13)

## Duplicate values in transactions

In [316]:
df_transactions.duplicated().sum()

np.int64(10)

In [317]:
# removing duplicate rows

In [318]:
df_transactions = df_transactions.drop_duplicates().reset_index(drop=True)

## Data types in transactions

In [319]:
df_transactions.dtypes

Transaction_ID         object
Transaction_Date       object
Customer_ID            object
Store_ID               object
Sales_Channel          object
Product_ID             object
Product_Name           object
Category               object
Quantity                int64
Unit_Price_INR        float64
Discount_Rate         float64
Payment_Method         object
Transaction_Status     object
dtype: object

In [320]:
# converting 'Transaction_Date' from object to datetime

In [321]:
df_transactions["Transaction_Date"] = pd.to_datetime(df_transactions["Transaction_Date"], format="%d-%m-%Y", errors='coerce')

## Missing values in transactions

In [322]:
df_transactions.isnull().sum()[df_transactions.isnull().sum()>0]

Transaction_Date    35
Customer_ID         80
dtype: int64

In [323]:
# 80 Customer_ID are missing. we will keep them for product, store and category sales. Exclude them from customer-level analysis

In [324]:
# Remove rows with missing transaction dates

In [325]:
df_transactions = df_transactions.dropna(
    subset=['Transaction_Date']
).reset_index(drop=True)

In [326]:
# Replace missing customer IDs with Unknown

In [327]:
df_transactions["Customer_ID"] = (df_transactions["Customer_ID"].fillna("Unknown"))

In [328]:
# verify

In [329]:
df_transactions.isnull().sum()[df_transactions.isnull().sum()>0]

Series([], dtype: int64)

## Text inconsistencies in transactions

In [330]:
# checking unique values in Sales_Channel
df_transactions["Sales_Channel"].unique()

array(['In-store', 'App', 'Online', 'APP', 'App ', 'online', 'IN-STORE'],
      dtype=object)

In [331]:
# remove extra spaces and standardize capitalization
df_transactions["Sales_Channel"] = (df_transactions["Sales_Channel"].str.strip().str.title())

In [332]:
# verify
df_transactions["Sales_Channel"].unique()

array(['In-Store', 'App', 'Online'], dtype=object)

In [333]:
# checking unique values in Product_Name
df_transactions["Product_Name"].unique()

array(['Sugar 1kg', 'Milk 1L', 'Wheat Flour 5kg', 'LED Bulb',
       'Laundry Detergent', 'Sunflower Oil 1L', 'Toothpaste', 'Curd 500g',
       'Shampoo', 'Tea 250g', 'Bread', 'Dishwash Liquid', 'Coffee 100g',
       'Eggs 12-pack', 'Biscuits Pack', 'Apples 1kg', 'Notebook',
       'Chocolate', 'Bananas 1kg', 'Rice 5kg'], dtype=object)

In [334]:
# it is ok

In [335]:
# checking unique values in Category
df_transactions["Category"].unique()

array(['Staples', 'Dairy', 'STAPLES', 'Home & Utility', 'Home Care',
       'Personal Care', 'Beverages', 'Bakery', 'HOME CARE',
       'Packaged Food', 'Fresh Produce', 'Stationery', 'PERSONAL CARE',
       'DAIRY', 'FRESH PRODUCE', 'STATIONERY', 'PACKAGED FOOD',
       'BEVERAGES', 'HOME & UTILITY'], dtype=object)

In [336]:
# remove extra spaces and standardize capitalization
df_transactions["Category"] = (df_transactions["Category"].str.strip().str.title())

In [337]:
# verify
df_transactions["Category"].unique()

array(['Staples', 'Dairy', 'Home & Utility', 'Home Care', 'Personal Care',
       'Beverages', 'Bakery', 'Packaged Food', 'Fresh Produce',
       'Stationery'], dtype=object)

In [338]:
# checking unique values in Payment_Method
df_transactions["Payment_Method"].unique()

array(['Debit Card', 'Cash', 'Wallet', 'UPI', 'Credit Card', 'Cash ',
       'upi', 'CREDIT CARD'], dtype=object)

In [339]:
# remove extra spaces and standardize capitalization
df_transactions["Payment_Method"] = (df_transactions["Payment_Method"].str.strip().str.title())

In [340]:
# verify
df_transactions["Payment_Method"].unique()

array(['Debit Card', 'Cash', 'Wallet', 'Upi', 'Credit Card'], dtype=object)

In [341]:
# checking unique values in Payment_Method
df_transactions["Transaction_Status"].unique()

array(['Completed', 'Cancelled', 'Returned', 'Pending', 'COMPLETE',
       'RETURN', 'completed', 'cancelled'], dtype=object)

In [342]:
# standardize capitalization
df_transactions["Transaction_Status"] = (df_transactions["Transaction_Status"].str.title())

In [343]:
# verify
df_transactions["Transaction_Status"].unique()

array(['Completed', 'Cancelled', 'Returned', 'Pending', 'Complete',
       'Return'], dtype=object)

In [344]:
df_transactions["Transaction_Status"] = df_transactions["Transaction_Status"].replace({'Complete':'Completed', 'Return':'Returned'})

In [345]:
# verify
df_transactions["Transaction_Status"].unique()

array(['Completed', 'Cancelled', 'Returned', 'Pending'], dtype=object)

## Checking for inconsistencies in numerical values

In [346]:
df_transactions.columns

Index(['Transaction_ID', 'Transaction_Date', 'Customer_ID', 'Store_ID',
       'Sales_Channel', 'Product_ID', 'Product_Name', 'Category', 'Quantity',
       'Unit_Price_INR', 'Discount_Rate', 'Payment_Method',
       'Transaction_Status'],
      dtype='object')

In [347]:
# checking unique values in Quantity
df_transactions["Quantity"].unique()

array([ 1,  2,  4,  3, -1])

In [348]:
# check how many negative values are there
(df_transactions["Quantity"] < 0).sum()

np.int64(13)

In [349]:
df_transactions[df_transactions["Quantity"] < 0]

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status
860,TX0000863,2025-09-08,DM001416,ST022,App,PR017,Coffee 100g,Beverages,-1,210.0,0.20,Upi,Completed
4703,TX0004712,2025-04-12,DM001127,ST009,In-Store,PR002,Wheat Flour 5kg,Staples,-1,245.0,0.00,Wallet,Completed
4855,TX0004864,2026-01-05,DM000415,ST029,In-Store,PR017,Coffee 100g,Beverages,-1,210.0,0.05,Debit Card,Completed
6801,TX0006814,2026-10-10,DM000562,ST028,App,PR011,Dishwash Liquid,Home Care,-1,115.0,0.00,Credit Card,Completed
7473,TX0007490,2025-06-07,DM001662,ST017,In-Store,PR014,Toothpaste,Personal Care,-1,99.0,0.02,Upi,Completed
8759,TX0008776,2025-09-29,DM003176,ST009,Online,PR002,Wheat Flour 5kg,Staples,-1,245.0,0.10,Debit Card,Completed
10279,TX0010300,2025-07-30,DM000194,ST013,In-Store,PR003,Sunflower Oil 1L,Staples,-1,145.0,0.20,Upi,Completed
10890,TX0010915,2026-08-25,DM003038,ST009,Online,PR009,Apples 1kg,Fresh Produce,-1,180.0,0.02,Wallet,Completed
11783,TX0011811,2025-03-03,DM000076,ST005,In-Store,PR004,Sugar 1kg,Staples,-1,48.0,0.00,Credit Card,Completed
12994,TX0013025,2026-08-30,DM003174,ST004,App,PR012,Laundry Detergent,Home Care,-1,299.0,0.15,Debit Card,Completed


In [350]:
# negative quantity values have 'Completed' Transaction_Status. It confirms that it is not for returned 
# hence we can replace it with positive value as it may be due to entry error
df_transactions["Quantity"] = df_transactions["Quantity"].abs()

In [351]:
# verify
df_transactions["Quantity"].unique()

array([1, 2, 4, 3])

In [352]:
# checking unique values in Unit_Price_INR
df_transactions["Unit_Price_INR"].unique()

array([  48.,   64.,  245.,   85.,  299.,  145.,   99.,   38.,  189.,
        155.,   42.,  115.,  210.,   92.,   30.,  180.,   55.,   50.,
         60.,  349., -210., -180.,  -60., -115.,  -42.,  -92., -349.,
        -85., -189.,  -50.])

In [353]:
# count of negative values
(df_transactions["Unit_Price_INR"] < 0).sum()

np.int64(12)

In [354]:
df_transactions[df_transactions["Unit_Price_INR"] < 0]

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status
376,TX0000378,2025-02-22,DM000604,ST026,Online,PR017,Coffee 100g,Beverages,1,-210.0,0.00,Wallet,Completed
689,TX0000692,2026-05-21,DM000680,ST003,Online,PR009,Apples 1kg,Fresh Produce,2,-180.0,0.05,Cash,Completed
1576,TX0001579,2025-11-07,DM002194,ST005,App,PR010,Bananas 1kg,Fresh Produce,2,-60.0,0.15,Credit Card,Completed
1675,TX0001678,2025-09-11,DM002814,ST019,In-Store,PR011,Dishwash Liquid,Home Care,1,-115.0,0.10,Debit Card,Completed
1908,TX0001912,2025-01-26,DM001297,ST010,Online,PR007,Bread,Bakery,2,-42.0,0.10,Debit Card,Pending
3968,TX0003976,2025-12-07,DM002675,ST010,In-Store,PR008,Eggs 12-pack,Dairy,1,-92.0,0.05,Credit Card,Completed
4950,TX0004959,2025-02-26,DM001252,ST016,App,PR010,Bananas 1kg,Fresh Produce,3,-60.0,0.05,Upi,Completed
8290,TX0008307,2025-04-15,DM002867,ST015,Online,PR001,Rice 5kg,Staples,1,-349.0,0.15,Cash,Completed
8413,TX0008430,2025-10-11,DM002959,ST009,App,PR007,Bread,Bakery,1,-42.0,0.10,Credit Card,Completed
9145,TX0009163,2025-10-29,DM000208,ST016,In-Store,PR020,LED Bulb,Home & Utility,1,-85.0,0.15,Credit Card,Completed


In [355]:
df_products[df_products["Product_ID"]=="PR017"]

,Product_ID,Product_Name,Category,List_Price_INR,Unit_Cost_INR
16,PR017,Coffee 100g,Beverages,210,150


In [356]:
df_products[df_products["Product_ID"]=="PR009"]

,Product_ID,Product_Name,Category,List_Price_INR,Unit_Cost_INR
8,PR009,Apples 1kg,Fresh Produce,180,120


In [357]:
# from above we can see that price in df_products and it is negative in transaction table due to entry error
# hence we can replace it with positive value
df_transactions["Unit_Price_INR"] = df_transactions["Unit_Price_INR"].abs()

In [358]:
# verify
df_transactions[df_transactions["Unit_Price_INR"] < 0]

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status


In [359]:
# # checking values less than 0 and greater than 1 in Discount_Rate
df_transactions[(df_transactions["Discount_Rate"] < 0) | (df_transactions["Discount_Rate"] > 1)]

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status
94,TX0000096,2026-04-23,DM001564,ST024,App,PR008,Eggs 12-pack,Dairy,1,92.0,1.2,Cash,Completed
1477,TX0001480,2025-10-16,DM001966,ST021,App,PR014,Toothpaste,Personal Care,1,99.0,1.2,Upi,Completed
2197,TX0002202,2026-06-08,DM003053,ST013,App,PR001,Rice 5kg,Staples,1,349.0,1.2,Upi,Completed
3027,TX0003033,2025-04-25,DM003042,ST017,Online,PR012,Laundry Detergent,Home Care,1,299.0,1.2,Credit Card,Completed
3801,TX0003809,2026-03-02,DM002963,ST017,Online,PR015,Biscuits Pack,Packaged Food,1,30.0,1.2,Debit Card,Completed
4297,TX0004305,2026-02-05,DM002842,ST021,In-Store,PR005,Milk 1L,Dairy,1,64.0,1.2,Wallet,Completed
6007,TX0006018,2025-04-22,DM001669,ST023,In-Store,PR017,Coffee 100g,Beverages,2,210.0,1.2,Wallet,Completed
8451,TX0008468,2025-11-28,DM000104,ST006,Online,PR002,Wheat Flour 5kg,Staples,1,245.0,1.2,Wallet,Completed
11898,TX0011929,2025-03-26,DM000767,ST028,In-Store,PR015,Biscuits Pack,Packaged Food,1,30.0,1.2,Wallet,Completed
12751,TX0012782,2026-04-11,DM000161,ST001,In-Store,PR015,Biscuits Pack,Packaged Food,1,30.0,1.2,Cash,Completed


In [360]:
# count
((df_transactions["Discount_Rate"] < 0) |
 (df_transactions["Discount_Rate"] > 1)).sum()

np.int64(10)

In [361]:
# discount can’t be 120%. We will replace it with Na
df_transactions.loc[df_transactions["Discount_Rate"] > 1,"Discount_Rate"] = np.nan

In [362]:
# verify
df_transactions[(df_transactions["Discount_Rate"] < 0) | (df_transactions["Discount_Rate"] > 1)]

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status


## Cross-check with the lookup

In [363]:
# Merge with the product lookup
df_check = df_transactions.merge(
    df_products[["Product_ID", "Product_Name", "Category"]],
    on="Product_ID",
    how="left",
    suffixes=("_transaction", "_lookup")
)

In [364]:
# Find mismatches
mismatches = df_check[
    (df_check["Product_Name_transaction"] != df_check["Product_Name_lookup"]) |
    (df_check["Category_transaction"] != df_check["Category_lookup"])
]

mismatches

,Transaction_ID,Transaction_Date,Customer_ID,Store_ID,Sales_Channel,Product_ID,Product_Name_transaction,Category_transaction,Quantity,Unit_Price_INR,Discount_Rate,Payment_Method,Transaction_Status,Product_Name_lookup,Category_lookup


In [365]:
# no mismatch found, no issue

# Add derived columns

In [366]:
# create gross sales column
df_transactions["Gross_Sales"] = (
    df_transactions["Quantity"] * df_transactions["Unit_Price_INR"]
)

In [367]:
# discount amount
df_transactions["Discount_Amount"] = (
    df_transactions["Gross_Sales"] * df_transactions["Discount_Rate"]
)

In [368]:
# net sales
df_transactions["Net_Sales"] = (
    df_transactions["Gross_Sales"] - df_transactions["Discount_Amount"]
)

In [369]:
# Estimated Cost

df_transactions = df_transactions.merge(
    df_products[["Product_ID", "Unit_Cost_INR"]],
    on="Product_ID",
    how="left"
)

df_transactions["Estimated_Cost"] = (
    df_transactions["Quantity"] * df_transactions["Unit_Cost_INR"]
)

In [370]:
# Estimated Gross Profit
df_transactions["Estimated_Gross_Profit"] = (
    df_transactions["Net_Sales"] - df_transactions["Estimated_Cost"]
)

In [371]:
# Extract Month
df_transactions["Month"] = df_transactions["Transaction_Date"].dt.month

In [372]:
# Extract Year
df_transactions["Year"] = df_transactions["Transaction_Date"].dt.year

In [373]:
# Create a Valid Rows Flag
df_transactions["Valid_Row"] = (
    df_transactions["Transaction_Date"].notna() &
    df_transactions["Quantity"].gt(0) &
    df_transactions["Unit_Price_INR"].ge(0) &
    df_transactions["Discount_Rate"].between(0, 1) &
    df_transactions["Unit_Cost_INR"].notna()
)

In [374]:
# check how many rows are valid versus invalid.
df_transactions["Valid_Row"].value_counts()

Valid_Row
True     14980
False       10
Name: count, dtype: int64

# Clean Customers Data

In [375]:
# look data
df_customers.head()

,Customer_ID,Customer_Name,Age,Gender,Mobile_Number,Email,City,State,Signup_Date,Customer_Segment,Loyalty_Tier,Preferred_Store_ID,Consent_for_Offers
0,DM000001,Neha Joshi,42.0,M,9724196932,neha.joshi0@example.com,Thane,Maharashtra,22-12-2024,Value Seeker,Silver,ST006,No
1,DM000002,Kiran Das,59.0,Female,9839365054,kiran.das1@example.com,Hyderabad,Telangana,28-09-2025,Premium,NaN,ST028,No
2,DM000003,Arjun Sharma,28.0,Female,9144751065,arjun.sharma2@example.com,Hyderabad,Telangana,09-02-2025,Occasional,Gold,ST006,No
3,DM000004,Sneha Singh,33.0,Female,9192470337,sneha.singh3@example.com,Bengaluru,Karnataka,25-09-2026,Occasional,Gold,ST009,Yes
4,DM000005,Aditya Rao,46.0,Female,9350282249,aditya.rao4@example.com,Mumbai,Maharashtra,10-10-2022,Premium,Silver,ST004,No


In [376]:
df_customers.shape

(3226, 13)

## checking data types

In [377]:
df_customers.dtypes

Customer_ID            object
Customer_Name          object
Age                   float64
Gender                 object
Mobile_Number          object
Email                  object
City                   object
State                  object
Signup_Date            object
Customer_Segment       object
Loyalty_Tier           object
Preferred_Store_ID     object
Consent_for_Offers     object
dtype: object

In [378]:
df_customers["Age"].unique()

array([ 42.,  59.,  28.,  33.,  46.,  29.,  18.,  40.,  32.,  nan,  35.,
        56.,  37.,  22.,  23.,  55.,  36.,  54.,  51.,  31.,  26.,  38.,
        44.,  64.,  27.,  49.,  65.,  25.,  34.,  43.,  20.,  21.,  45.,
        52.,  48.,  57.,  19.,  30.,  41.,  50.,  24.,  39.,  47.,  53.,
        63.,  67.,  60.,  74.,  58.,  62.,  70.,  68.,   5.,  -3.,  73.,
        61., 121.,  69.,  66.,  78.,  71.,  72.,  76.,  77., 250.])

In [379]:
# age should be int as per data
df_customers["Age"] = df_customers["Age"].astype("Int64")

In [380]:
# fix signup_date format
df_customers["Signup_Date"] = pd.to_datetime(
    df_customers["Signup_Date"],
    errors="coerce"
)

In [381]:
# verify
df_customers.dtypes

Customer_ID                   object
Customer_Name                 object
Age                            Int64
Gender                        object
Mobile_Number                 object
Email                         object
City                          object
State                         object
Signup_Date           datetime64[ns]
Customer_Segment              object
Loyalty_Tier                  object
Preferred_Store_ID            object
Consent_for_Offers            object
dtype: object

## Text inconsistencies in customers

In [382]:
df_customers.columns

Index(['Customer_ID', 'Customer_Name', 'Age', 'Gender', 'Mobile_Number',
       'Email', 'City', 'State', 'Signup_Date', 'Customer_Segment',
       'Loyalty_Tier', 'Preferred_Store_ID', 'Consent_for_Offers'],
      dtype='object')

In [383]:
# standardise the text columns
df_customers["Customer_Name"] = (df_customers["Customer_Name"].str.strip().str.title())
df_customers["Gender"] = (df_customers["Gender"].str.strip().str.title())
df_customers["Email"] = (df_customers["Email"].str.strip().str.lower())
df_customers["City"] = (df_customers["City"].str.strip().str.title())
df_customers["Loyalty_Tier"] = (df_customers["Loyalty_Tier"].str.strip().str.title())
df_customers["Consent_for_Offers"] = (df_customers["Consent_for_Offers"].str.strip().str.title())

In [384]:
# checking gender values
df_customers["Gender"].unique()

array(['M', 'Female', 'Male', 'Prefer Not To Say', 'F', nan], dtype=object)

In [385]:
# we need to replace:
# 'M' with 'Male', 
#'F' with 'Female'

df_customers["Gender"] = df_customers["Gender"].replace({'M':'Male', 'F':'Female'})

In [386]:
# verify
df_customers["Gender"].unique()

array(['Male', 'Female', 'Prefer Not To Say', nan], dtype=object)

In [387]:
# checking city values
df_customers["City"].unique()

array(['Thane', 'Hyderabad', 'Bengaluru', 'Mumbai', 'Chennai', 'Pune',
       'Kolkata', 'Nagpur', 'Delhi', 'Ahmedabad'], dtype=object)

In [388]:
# ok

## Checking for duplicate rows

In [389]:
# count of duplicated rows
df_customers.duplicated().sum()

np.int64(26)

In [390]:
# removing duplicated rows
df_customers = df_customers.drop_duplicates().reset_index(drop=True)

In [391]:
# verify
df_customers.duplicated().sum()

np.int64(0)

## Check for missing values

In [392]:
df_customers.isnull().sum()[df_customers.isnull().sum()>0]

Age                  64
Gender                7
Mobile_Number        43
Email                40
Signup_Date          24
Customer_Segment      4
Loyalty_Tier        792
dtype: int64

In [397]:
# Invalid Ages
df_customers[(df_customers["Age"]<10) | (df_customers["Age"]>100)]

,Customer_ID,Customer_Name,Age,Gender,Mobile_Number,Email,City,State,Signup_Date,Customer_Segment,Loyalty_Tier,Preferred_Store_ID,Consent_for_Offers
832,DM000833,Pooja Kumar,5,Male,9271710325,pooja.kumar56@example.com,Chennai,Tamil Nadu,2023-08-01,Value Seeker,Gold,ST026,No
845,DM000846,Amit Jain,-3,Male,9095382779,amit.jain69@example.com,Hyderabad,Telangana,2025-04-18,Premium,Silver,ST014,Yes
1164,DM001165,Kavya Rao,121,Male,9294276042,kavya.rao0@example.com,Hyderabad,Telangana,2022-05-01,Premium,Gold,ST005,Yes
1188,DM001189,Ishaan Sharma,5,Prefer Not To Say,9673503814,ishaan.sharma24@example.com,Bengaluru,Karnataka,2022-03-09,Regular,Gold,ST024,No
1851,DM001852,Aarav Joshi,-3,Male,9032445315,aarav.joshi8@example.com,Ahmedabad,Gujarat,2025-06-22,Value Seeker,Platinum,ST005,No
1952,DM001953,Rahul Kumar,-3,Male,9830501763,rahul.kumar12@example.com,Mumbai,Maharashtra,2023-05-16,Family Shopper,Silver,ST023,No
2433,DM002434,Ira Das,250,Male,9075127331,ira.das8@example.com,Ahmedabad,Gujarat,2026-03-22,Occasional,Platinum,ST016,Yes
2475,DM002476,Pooja Jain,250,Female,9604302136,pooja.jain50@example.com,Hyderabad,Telangana,2023-10-22,Regular,Platinum,ST025,No
2949,DM002950,Arjun Mishra,250,Female,9774874566,arjun.mishra39@example.com,Hyderabad,Telangana,2026-05-30,Premium,Silver,ST015,No
3090,DM003091,Riya Patel,121,NaN,9409194901,riya.patel83@example.com,Ahmedabad,Gujarat,2023-11-06,Occasional,Platinum,ST011,Yes


In [398]:
# replacing invalid ages with na
df_customers["Age"] = df_customers["Age"].mask((df_customers["Age"] < 10) | (df_customers["Age"] > 100), np.nan)

In [400]:
# set gender to "Unknown" for na
df_customers["Gender"] = df_customers["Gender"].fillna("Unknown")

In [401]:
# set Customer_Segment to "Unknown" for na
df_customers["Customer_Segment"] = df_customers["Customer_Segment"].fillna("Unknown")

In [402]:
# set Loyalty_Tier to "Unknown" for na
df_customers["Loyalty_Tier"] = df_customers["Loyalty_Tier"].fillna("Unknown")

In [ ]:
# keep Mobile_Number, Email,  as it is, it does not affect analysis
# we can not gues for Signup_date, leave it as it is

# Clean Products Data

In [404]:
# look data
df_products.head()

,Product_ID,Product_Name,Category,List_Price_INR,Unit_Cost_INR
0,PR001,Rice 5kg,Staples,349,275
1,PR002,Wheat Flour 5kg,Staples,245,180
2,PR003,Sunflower Oil 1L,Staples,145,112
3,PR004,Sugar 1kg,Staples,48,36
4,PR005,Milk 1L,Dairy,64,52


In [405]:
df_products.shape

(20, 5)

In [407]:
# checking data types
df_products.dtypes

Product_ID        object
Product_Name      object
Category          object
List_Price_INR     int64
Unit_Cost_INR      int64
dtype: object

In [406]:
# check unique values in Product_Name
df_products["Product_Name"].unique()

array(['Rice 5kg', 'Wheat Flour 5kg', 'Sunflower Oil 1L', 'Sugar 1kg',
       'Milk 1L', 'Curd 500g', 'Bread', 'Eggs 12-pack', 'Apples 1kg',
       'Bananas 1kg', 'Dishwash Liquid', 'Laundry Detergent', 'Shampoo',
       'Toothpaste', 'Biscuits Pack', 'Tea 250g', 'Coffee 100g',
       'Chocolate', 'Notebook', 'LED Bulb'], dtype=object)

In [408]:
# check unique values in Category
df_products["Category"].unique()

array(['Staples', 'Dairy', 'Bakery', 'Fresh Produce', 'Home Care',
       'Personal Care', 'Packaged Food', 'Beverages', 'Stationery',
       'Home & Utility'], dtype=object)

In [410]:
# check for price less than 0
df_products[df_products["List_Price_INR"]<0]

,Product_ID,Product_Name,Category,List_Price_INR,Unit_Cost_INR


In [411]:
# check for Unit_Cost_INR less than 0
df_products[df_products["Unit_Cost_INR"]<0]

,Product_ID,Product_Name,Category,List_Price_INR,Unit_Cost_INR


In [ ]:
# Products data is fine with no issues

# Validate relationships

## Check customer IDs in both directions

In [413]:
# Transactions → Customers
valid_customer_ids = set(df_customers["Customer_ID"].dropna())

unmatched_transactions = df_transactions[
    ~df_transactions["Customer_ID"].isin(valid_customer_ids)
    & df_transactions["Customer_ID"].ne("Unknown")
]

print("Transactions with unmatched Customer_ID:",
      len(unmatched_transactions))

Transactions with unmatched Customer_ID: 0


In [414]:
# Customers → Transactions
transaction_customer_ids = set(
    df_transactions["Customer_ID"].dropna()
)

customers_without_transactions = df_customers[
    ~df_customers["Customer_ID"].isin(transaction_customer_ids)
]

print("Customers without transactions:",
      len(customers_without_transactions))

Customers without transactions: 31


## Check transactions before signup

In [415]:
df_check = df_transactions.merge(
    df_customers[["Customer_ID", "Signup_Date"]],
    on="Customer_ID",
    how="left"
)

before_signup = df_check[
    (df_check["Customer_ID"] != "Unknown") &
    df_check["Signup_Date"].notna() &
    (df_check["Transaction_Date"] < df_check["Signup_Date"])
]

print("Transactions before signup:", len(before_signup))
print("Percentage:", round(len(before_signup) / len(df_transactions) * 100, 2), "%")

Transactions before signup: 2674
Percentage: 17.84 %


## Check preferred-store matches

In [416]:
df_store_check = df_transactions.merge(
    df_customers[["Customer_ID", "Preferred_Store_ID"]],
    on="Customer_ID",
    how="left"
)

store_matches = df_store_check[
    (df_store_check["Customer_ID"] != "Unknown") &
    df_store_check["Preferred_Store_ID"].notna() &
    (df_store_check["Store_ID"] == df_store_check["Preferred_Store_ID"])
]

print("Transactions at preferred store:", len(store_matches))
print("Percentage:", round(len(store_matches) / len(df_transactions) * 100, 2), "%")

Transactions at preferred store: 539
Percentage: 3.6 %


# Save Cleaned Files

In [423]:
# save 3 cleaned csv's
df_transactions.to_csv("C:/Users/Care/Desktop/Dmart_Project/data/clean/transactions_clean.csv", index=False)
df_customers.to_csv("C:/Users/Care/Desktop/Dmart_Project/data/clean/customers_clean.csv", index=False)
df_products.to_csv("C:/Users/Care/Desktop/Dmart_Project/data/clean/products_clean.csv", index=False)